In [ ]:
!pip install bertopic datasets openai datamapplot

# 零、总述

本章主要探讨生成模型和表示模型如何在**无监督学习**领域发挥作用，尽管有监督算法比较盛行，但**无监督的方法由于能够在无需提前标注的情况下基于语义内容对文本进行分组，仍然具有巨大潜力**

下面要介绍的**文本聚类（Text Clustering）** 和 **主题建模(Topic Modeling)** 就是属于无监督学习算法

*文本聚类* 旨在基于文本的语义内容、含义和关系对相似文本进行分组

<img src="./resources/text_clustering.png">


*主题建模* 旨在找出这些文本都在讨论哪些主题，所以它关注的问题是：一个文本集合中，隐藏着哪些主题？每个文本又包含哪些主题？

<img src="./resources/topic_modeling.png">



首先，我们先探索如何使用嵌入模型进行聚类，然后过渡到一种受文本聚类启发的主题建模方法，即 BERTopic。

在本章中，我们将在 ArXiv 文章上运行聚类和主题建模算法。ArXiv 是一个主要面向计算机科学、数学和物理领域的开放的学术文章平台。下面我们将探索计算与语言领域的文章，对应于 arxiv_nlp 这一数据集，它包含 1991 年至 2024 年间来自 ArXiv cs.CL 板块的 44949 篇摘要。

下面我们加载数据，做数据准备

In [ ]:
from datasets import load_dataset
dataset = load_dataset("maartengr/arxiv_nlp")["train"]

abstracts = dataset["Abstracts"]
titles = dataset["Titles"]

# 一、文本聚类的通用流程

文本聚类不仅可以发现已知的数据模式，更可以挖掘不为人知的数据模式，它可以帮助你直观地理解任务及其复杂性。虽然文本聚类的方法有很多，从基于图的神经网络到基于质心的聚类技术，但当前比较流行的通用流程主要包含以下三个步骤（Embedding -> Reduction -> Cluster）：
1. 使用**嵌入模型（Embedding Model）** 将输入文档转换为嵌入向量
2. 使用**降维模型（Dimensionality Reduction Model）** 将嵌入向量降至更低维度空间
3. 使用**聚类模型（Cluster Model）** 对降维后的嵌入向量进行聚类

## 1.1 嵌入文档

这里我们选择 thenlper/gte-small 模型来作为嵌入模型

In [ ]:
from sentence_transformers import SentenceTransformer

# 为每个摘要创建嵌入向量
embedding_model = SentenceTransformer("thenlper/gte-small")
embeddings = embedding_model.encode(abstracts, show_progress_bar=True)

In [ ]:
embeddings.shape

我们可以看到嵌入向量的维度为 384 维，即每个摘要的语义表示（向量表示）包含了 384 个值

## 1.2 嵌入向量降维

降维：高维空间 -> 低维空间

**降维** 会带来两个问题：第一个是计算量会变得很大，但最主要的还是第二个：**高维空间会使数据变得非常“稀疏”**，也就是说，**维度越高，点之间距离越远**，其原理基于欧式距离的公式，公式如下：
$$
D = \sqrt{(a_1 - b_1)^2 + (a_2 - b_2)^2 + \cdot + (a_d - b_d)^2}
$$

因此，**距离会随着维度的平方根增长**，并且会带来一个很重要的问题：**距离的区分能力下降**，也就是*最近的点和最远的点之间，相对差异越来越小*，例如低维空间：
```text
最近邻：0.2
最远点：3.0
```

但是到达了高维空间之后，它会变成：
```text
最近邻：400
最远点：430
```

所以，**降维**不仅是为了减少计算量，更重要的是把数据从过于稀疏的高维空间压缩到一个更紧凑、更有意义的低维表示空间中。还有一点需要补充的是：**维度不是越低越好**，那样会丢失大量的有用信息，所以要在信息保留与降维之间做权衡。

我们介绍两个降维的方法：主成分分析（Principal Component Analysis,**PCA**）和统一流行逼近和投影(Uniform Manifold Approximation and Projection, **UMAP**)。

1. PCA: 主成分分析

它是一种经典的**线性降维算法**。PCA 将数据投影到一组新的正交坐标轴上，这些轴按照能够解释的数据方差从大到小排列。它核心的思想是**找到数据变化最大的方向，然后把数据投影到这些方向上**

2. UMAP：统一流行逼近和投影

它是一种**非线性降维算法/流形学习算法**，它更关注“谁和谁是邻居”，它的核心目标之一是**让低维空间尽量保持高维数据中的邻域结构**

本次流程我们使用的是 UMAP，因为它在处理非线性关系和结构方面比 PCA 表现更好

In [ ]:
from umap import UMAP

# 将嵌入向量从 384 维降至 5 维
umap_model = UMAP(
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

reduced_embeddings = umap_model.fit_transform(embeddings)

## 1.3 对降维后的嵌入向量进行聚类

这里我们先来介绍一些聚类算法



这里，我们使用算法 HDBSCAN 来对上述嵌入向量进行聚类

In [ ]:
from hdbscan import HDBSCAN

# 拟合模型并提取簇
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    metric="euclidean",
    cluster_selection_method="eom"
).fit(reduced_embeddings)

clusters = hdbscan_model.labels_

print(f"生成簇的数量为：{len(set(clusters))}")

## 1.4 检查生成的簇

# 二、从文本聚类到主题建模

## 2.1 BERTopic: 一个模块化的主题建模框架

## 2.2 添加特殊的 “乐高积木块”

## 2.3 文本生成的 “乐高积木块”